In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import urllib.request
from tqdm import tqdm
import os
import json

In [6]:
keypoint_selection = "vitpose" # options: "coco", "vitpose"

In [7]:
keypoint_data = np.load(f"../{keypoint_selection}/keypoints_data.npy", allow_pickle=True)
coco_items = np.load("../coco_orientations.npy")

In [12]:
len(coco_items)

18769

In [5]:
def add_json_item(id, image, prompt, response):
    return {
        "id": id,
        "image": image,
        "conversations": [
            {
                "from": "human",
                "value": f"<image>\n{prompt}"
            },
            {
                "from": "gpt",
                "value": response
            }
        ]
    }

In [6]:
if keypoint_selection == "vitpose":  # extra data cleaning
    num_items = keypoint_data.shape[0]

    zero_count = 0
    for i in range(len(keypoint_data)):
        if np.sum(keypoint_data[i]) == 0:
            zero_count += 1
    assert zero_count == keypoint_data.shape[0] - num_items

    # Identify rows where all columns except the first are NaN
    nan_rows = np.isnan(keypoint_data).all(axis=1)
    # Print indices of rows to be removed
    nan_indices = np.where(nan_rows)[0]
    print("Indices of rows to be removed:", nan_indices)
    # Remove these rows from keypoint_data
    keypoint_data = keypoint_data[~nan_rows]
    print("Keypoint data shape after removing NaN rows:", keypoint_data.shape)

    # add coco IDs as first column to keypoint_data (data_items[:, 0])
    coco_ids = coco_items[:, 0]
    print("Coco IDs before nan removal:", coco_ids.shape)
    coco_ids = coco_ids[~nan_rows]
    print("Coco IDs after nan removal:", coco_ids.shape)
    coco_ids = coco_ids.reshape(-1, 1)
    print("Coco IDs reshaped:", coco_ids.shape)

    keypoint_data = np.hstack((coco_ids, keypoint_data))

### check body keypoints

In [7]:
def load_image(image_id):
    image = Image.open(
        urllib.request.urlopen(
            f'http://images.cocodataset.org/train2017/{image_id:012d}.jpg')
            ).convert("RGB")
    # make image 336x336
    image = image.resize((336, 336))
    return image

In [ ]:
# choose random number in num_items range
rand_num = np.random.randint(num_items)
image_id = keypoint_data[rand_num][0]
image = load_image(image_id)
# display image
print(keypoint_data[rand_num][1:])
plt.imshow(image)
plt.title(f"Image ID: {image_id}\n Keypoints: {keypoint_data[rand_num][1:]}")
plt.axis('off')

### Create keypoint/pose tokens

In [8]:
import math

IMG_W = IMG_H = 336                    # your resized frame
CONF_BUCKETS = 10                      # CONF_0 … CONF_9
YAW_BINS     = 8                       # 45° each   → YAW_0 … YAW_7
TORSO_EDGES  = (10, 25, 50)            # px cut-offs for TORSO_W_0-3

def _coord_token(val, axis):
    """Clamp to image bounds and return 'X_123' / 'Y_045' token."""
    v = int(round(np.clip(val, 0, IMG_W-1)))
    return f"{axis}_{v}"

def _conf_token(score):
    """ViTPose gives 0-1 float; bucket into CONF_0 … CONF_9."""
    bucket = min(CONF_BUCKETS-1, int(score * CONF_BUCKETS))
    return f"CONF_{bucket}"

def _yaw_token(dx, dy, text=False):
    """8 bins over 360°.  Image y-axis points *down*, so flip sign."""
    yaw_deg = (math.degrees(math.atan2(-dy, dx)) + 360) % 360
    bucket  = int(yaw_deg // (360 / YAW_BINS))
    if text:
        return str(bucket)
    else:
        return f"YAW_{bucket}"

def _torso_w_token(dist, text=False):
    """4 coarse bins: 0-10, 11-25, 26-50, >50 px."""
    if   dist <= TORSO_EDGES[0]: return "TORSO_W_0" if not text else "0"
    elif dist <= TORSO_EDGES[1]: return "TORSO_W_1" if not text else "1"
    elif dist <= TORSO_EDGES[2]: return "TORSO_W_2" if not text else "2"
    else:                        return "TORSO_W_3" if not text else "3"

In [ ]:
def build_pose_tokens(keypoints, img_w=336, img_h=336, keypoint_selection="coco"):
    """
    flat12 : iterable length-12  (strings or floats)
             [LSx, LSy, LSconf, RSx, RSy, RSconf, LHx, LHy, LHconf, RHx, RHy, RHconf]
    returns : list[str]          token sequence
    """
    if keypoint_selection == "coco":
        lsh_x, lsh_y = keypoints[1:3]
        rsh_x, rsh_y = keypoints[3:5]
        lhp_x, lhp_y = keypoints[5:7]
        rhp_x, rhp_y = keypoints[7:9]

        toks = [
            "POSE_START",
            "SHOULDER_L", _coord_token(lsh_x, "X"), _coord_token(lsh_y, "Y"),
            "SHOULDER_R", _coord_token(rsh_x, "X"), _coord_token(rsh_y, "Y"),
            "HIP_L",      _coord_token(lhp_x, "X"), _coord_token(lhp_y, "Y"),
            "HIP_R",      _coord_token(rhp_x, "X"), _coord_token(rhp_y, "Y"),
        ]
    elif keypoint_selection == "vitpose":
        f = list(map(float, keypoints))
        lsh_x, lsh_y, lsh_c = f[1:4] 
        rsh_x, rsh_y, rsh_c = f[4:7] 
        lhp_x, lhp_y, lhp_c = f[7:10] 
        rhp_x, rhp_y, rhp_c = f[10:13]

        toks = [
            "POSE_START",
            "SHOULDER_L", _coord_token(lsh_x, "X"), _coord_token(lsh_y, "Y"), _conf_token(lsh_c),
            "SHOULDER_R", _coord_token(rsh_x, "X"), _coord_token(rsh_y, "Y"), _conf_token(rsh_c),
            "HIP_L",      _coord_token(lhp_x, "X"), _coord_token(lhp_y, "Y"), _conf_token(lhp_c),
            "HIP_R",      _coord_token(rhp_x, "X"), _coord_token(rhp_y, "Y"), _conf_token(rhp_c),
        ]  

    # helpers need globals IMG_W / IMG_H; update if you pass custom size
    global IMG_W, IMG_H
    IMG_W, IMG_H = img_w, img_h
      

    # orientation bundle ---------------------------------------------------
    dx, dy = rsh_x - lsh_x, rsh_y - lsh_y
    yaw_tok   = _yaw_token(dx, dy)
    torso_tok = _torso_w_token(math.hypot(dx, dy))

    toks += ["ORIENT_START", yaw_tok, torso_tok, "ORIENT_END", "POSE_END"]
    return toks

In [ ]:
print(keypoint_data[rand_num])
print(build_pose_tokens(keypoint_data[rand_num]))

### Set up atomic prompts

In [ ]:
import random

train_data = []
prompt_options = ["Estimate this person\u0027s pose.",
                  "Return the pose-estimation tokens for the person in the picture.",
                  "Give me the discrete pose of the subject (shoulders & hips).",
                  "Provide the shoulder/hip key-points as tokens.",
                  "Output this person\u0027s pose in token form.",
                  "What are the pose tokens for the individual shown?"]
weights = [0.50, 0.25, 0.07, 0.06, 0.06, 0.06]
def pick_prompt() -> str:
    """Return one prompt string, sampled with the given weights."""
    return random.choices(prompt_options, weights=weights, k=1)[0]

In [ ]:
def add_ds_item(id_value, pose_data, prompt_options=prompt_options, weights=weights):
    """Add a single item to the dataset."""
    prompt = pick_prompt()

    coco_id = pose_data[0]
    pose_tokens = build_pose_tokens(pose_data)

    wrapped_tokens = [f"<{tok}>" for tok in pose_tokens]
    response = "".join(wrapped_tokens)

    # print("Pose Encoded Tokens:", response)

    return add_json_item(id_value, f"{coco_id}.jpg", prompt, f"{response}")

In [ ]:
add_ds_item('01', keypoint_data[rand_num])

### Format data

In [ ]:
output_path = f"../{keypoint_selection}/curriculum_data/atomic_items.json"
data_length = 14000 if keypoint_selection == "coco" else 18000
keypoint_data = keypoint_data[:data_length]

In [ ]:
# Initialize empty file if it doesn't exist
if not os.path.exists(output_path):
    with open(output_path, "w") as f:
        json.dump([], f)
    existing_data = []
else:
    with open(output_path, "r") as f:
        existing_data = json.load(f)

print(len(existing_data))

train_data = []

for i, item in tqdm(enumerate(keypoint_data)):
    try:
        # id will be the index of the item in the dataset with leading zeroes
        id_value = str(i + 1).zfill(5)  # 5 digits with leading zeroes
        ds_item = add_ds_item(id_value, item)
    except Exception as e:
        print(f"Error: {e}")
    
    train_data.append(ds_item)

    # Append to JSON every 100 iterations
    if (i + 1) % 100 == 0 or i == len(keypoint_data) - 1:
        # Read existing data
        with open(output_path, "r") as f:
            existing_data = json.load(f)

        # Append new data
        existing_data.extend(train_data)

        # Write back
        with open(output_path, "w") as f:
            json.dump(existing_data, f, indent=4)

        # Clear train_data
        train_data = []

### Text tokens

In [ ]:
def build_text_tokens(keypoints, img_w=336, img_h=336, keypoint_selection="coco"):
    """
    flat12 : iterable length-12  (strings or floats)
             [LSx, LSy, LSconf, RSx, RSy, RSconf, LHx, LHy, LHconf, RHx, RHy, RHconf]
    returns : list[str]          token sequence
    """
    # --- cast to float once ----------------------------------------------
    if keypoint_selection == "coco":
        lsh_x, lsh_y = keypoints[1:3]
        rsh_x, rsh_y = keypoints[3:5]
        lhp_x, lhp_y = keypoints[5:7]
        rhp_x, rhp_y = keypoints[7:9]

        toks = [lsh_x, lsh_y, rsh_x, rsh_y, lhp_x, lhp_y, rhp_x, rhp_y]
    elif keypoint_selection == "vitpose":
        f = list(map(float, keypoints))
        lsh_x, lsh_y, lsh_c = f[1:4] 
        rsh_x, rsh_y, rsh_c = f[4:7] 
        lhp_x, lhp_y, lhp_c = f[7:10] 
        rhp_x, rhp_y, rhp_c = f[10:13]

        toks = [lsh_x, lsh_y, lsh_c, rsh_x, rsh_y, rsh_c, lhp_x, lhp_y, lhp_c, rhp_x, rhp_y, rhp_c]
    toks = [str(tok) for tok in toks]

    # helpers need globals IMG_W / IMG_H; update if you pass custom size
    global IMG_W, IMG_H
    IMG_W, IMG_H = img_w, img_h

    # orientation bundle ---------------------------------------------------
    dx, dy = rsh_x - lsh_x, rsh_y - lsh_y
    yaw_tok   = _yaw_token(dx, dy, text=True)
    torso_tok = _torso_w_token(math.hypot(dx, dy), text=True)

    toks += [yaw_tok, torso_tok]
    return toks

In [ ]:
def add_text_item(id_value, pose_data, prompt_options=prompt_options, weights=weights):
    """Add a single item to the dataset."""
    prompt = pick_prompt()

    coco_id = pose_data[0]
    pose_tokens = build_text_tokens(pose_data)

    response = " ".join(pose_tokens)

    # print("Pose Encoded Tokens:", response)

    return add_json_item(id_value, f"{coco_id}.jpg", prompt, f"{response}")

In [ ]:
text_output_path = f"../{keypoint_selection}/curriculum_data/atomic_items_text.json"
text_data_length = 14000 if keypoint_selection == "coco" else 18000
keypoint_data = keypoint_data[:text_data_length]

# Initialize empty file if it doesn't exist
if not os.path.exists(text_output_path):
    with open(text_output_path, "w") as f:
        json.dump([], f)
    existing_text_data = []
else:
    with open(text_output_path, "r") as f:
        existing_text_data = json.load(f)

print(len(existing_text_data))

text_train_data = []

for i, item in tqdm(enumerate(keypoint_data)):
    try:
        # id will be the index of the item in the dataset with leading zeroes
        id_value = str(i + 1).zfill(5)  # 5 digits with leading zeroes
        ds_item = add_text_item(id_value, item)
    except Exception as e:
        print(f"Error: {e}")
    
    text_train_data.append(ds_item)

    # Append to JSON every 100 iterations
    if (i + 1) % 100 == 0 or i == len(keypoint_data) - 1:
        # Read existing data
        with open(text_output_path, "r") as f:
            existing_text_data = json.load(f)

        # Append new data
        existing_text_data.extend(text_train_data)

        # Write back
        with open(text_output_path, "w") as f:
            json.dump(existing_text_data, f, indent=4)

        # Clear train_data
        text_train_data = []